## 1) 设定数据路径并检查数据形状
这一格会选择数据集（`DATA_NAME`），并读取 `train/val/test` 的基本信息，先确认维度是否符合预期。


In [1]:
import os
from pathlib import Path

# Set CPU math thread limits before importing numpy/torch-backed libraries.
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"

import h5py
import numpy as np

DATA_NAME = "SEED"
AUTODL_DATA_ROOT = Path(f"/root/autodl-tmp/MLproject/course_project/{DATA_NAME}")
LOCAL_DATA_ROOT = Path.cwd() / "course_project" / DATA_NAME
DATA_ROOT = AUTODL_DATA_ROOT if AUTODL_DATA_ROOT.exists() else LOCAL_DATA_ROOT

INDEX_PATH_TRAIN = str(DATA_ROOT / "train.h5")
INDEX_PATH_VAL = str(DATA_ROOT / "val.h5")
INDEX_PATH_TEST = str(DATA_ROOT / "test_x_only.h5")

print(f"DATA_ROOT: {DATA_ROOT}")
with h5py.File(INDEX_PATH_TRAIN, "r") as f:
    print("keys:", list(f.keys()))
    print("x dtype:", f["X"].dtype)
    print("x shape:", f["X"].shape)
    print("y dtype:", f["y"].dtype)
    print("y shape:", f["y"].shape)
    y = f["y"][()]
    print("unique:", np.unique(y))


DATA_ROOT: /root/autodl-tmp/MLproject/course_project/SEED
keys: ['X', 'y']
x dtype: float32
x shape: (900, 62, 400)
y dtype: int64
y shape: (900,)
unique: [0 1 2]


## 2) 定义模型：SimpleLinear
最简单的线性分类器，方便和更复杂模型对比。


In [2]:
import torch
import torch.nn as nn

class SimpleLinear(nn.Module):
    def __init__(self, input_channels, time_points, num_classes):
        super(SimpleLinear, self).__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(input_channels * time_points, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        return self.fc(x)


## 3) 定义模型：SimpleMLP
把输入拉平后通过多层全连接网络进行分类，表达能力比线性模型更强。


In [3]:
class SimpleMLP(nn.Module):
    def __init__(
        self,
        input_channels,
        num_classes,
        time_points=200,
        hidden_dims=(256, 128),
        dropout=0.3
    ):
        super().__init__()

        input_dim = input_channels * time_points

        layers = []
        prev_dim = input_dim

        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = h

        layers.append(nn.Linear(prev_dim, num_classes))

        self.flatten = nn.Flatten()
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        # x: (B, C, T)
        x = self.flatten(x)      # -> (B, C*T)
        logits = self.mlp(x)     # -> (B, num_classes)
        return logits

## 4) 定义模型：EEGNet
EEGNet 的本质是一个“结构上受约束的 CNN”，通过“时间卷积 + 空间卷积（depthwise）+ 可分离卷积（separable）”分阶段提取 EEG 的时域、频域和空间特征。

卷积结构版本，利用时序与通道方向的局部模式，适合 EEG 信号特征提取。常用于作为baseline


In [4]:
class EEGNet(nn.Module):  # EEGNet-8,2
    def __init__(self, chans,time_point=200,f1=8, d=2, pk1=4, pk2=8, dp=0.5, max_norm1=1,norm=torch.nn.Identity()):
        super(EEGNet, self).__init__()
        f2 = f1 * d
        self.block1 = nn.Sequential(
            nn.Conv2d(1, f1, (1, 64), padding=(0,32), bias=False),
            nn.BatchNorm2d(f1),
        )
        # Spatial Filters
        self.block2 = nn.Sequential(
            nn.Conv2d(f1, d * f1, (chans, 1), groups=f1, bias=False),  # Depthwise Conv
            nn.BatchNorm2d(d * f1),
            nn.ELU(),
            nn.AvgPool2d((1, pk1), stride=pk1),
            nn.Dropout(dp)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(d * f1, f2, (1, 16), groups=f2, bias=False, padding=(0,8)),  # Separable Conv
            nn.Conv2d(f2, f2, kernel_size=1, bias=False),  # Pointwise Conv
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d((1, pk2), stride=pk2),
            nn.Dropout(dp)
        )

        self._apply_max_norm(self.block2[0], max_norm1)
        self.embed_dim = f2 * ((time_point // pk1) // pk2)
        self.norm=norm


    def _apply_max_norm(self, layer, max_norm):
        for name, param in layer.named_parameters():
            if 'weight' in name:
                param.data = torch.renorm(param.data, p=2, dim=0, maxnorm=max_norm)

    def forward(self, x):
        self.norm(x)
        if len(x.shape) == 2:
            x = x.unsqueeze(dim=1)
        x = self.block1(x.unsqueeze(dim=1))
        x = self.block2(x)
        x = self.block3(x)
        return x.flatten(start_dim=1)

## 5) 导入模型：EEGGRU
这里希望同学们自己手搓一个RNN代码，试试RNN的训练效果如何


In [5]:
# GPU environment configuration
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Enable TF32 matmul/convolution on Ampere/Ada/Blackwell GPUs for faster training.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

from RNN import EEGGRU


Using device: cuda:0
GPU name: NVIDIA GeForce RTX 5090


## 6) 多种子网格搜索 EEGGRU 最佳参数
这一格会固定随机种子，对 EEGGRU 参数网格进行多次训练；按参数组的平均验证准确率选择最佳参数，并保存完整结果、聚合结果、最佳参数和最终 test 预测 txt。


In [ ]:
import csv
import json
import math
import random
from itertools import product
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from course_project.TEST_DATASET import TrainDataset, TestDataset


# -------------------------
# Search configuration
# -------------------------
CHANNELS = 62
PATCH_SIZE = 400
CLASSES = 3
BATCH_SIZE = 32
EPOCHS = 100
PATIENCE = 20
MIN_DELTA = 0.0
LABEL_SMOOTHING = 0.0
BIDIRECTIONAL = True
NUM_WORKERS = 4
FORCE_RERUN = False
DEBUG_MODE = False
RUN_GRID_SEARCH = True
DETERMINISTIC = True
USE_CUDA = device.type == "cuda"

SEEDS = [42, 3407, 2024, 2025, 2026]
PARAM_GRID = {
    "hidden_dim": [32, 64, 128],
    "num_layers": [1, 2],
    "dropout": [0.3, 0.5],
    "lr": [1e-3, 5e-4],
    "weight_decay": [1e-3, 5e-3],
}

if DEBUG_MODE:
    EPOCHS = 1
    PATIENCE = 1
    SEEDS = SEEDS[:1]
    PARAM_GRID = {key: values[:1] for key, values in PARAM_GRID.items()}
    NUM_WORKERS = 0

SEARCH_ROOT = Path(DATA_ROOT) / "grid_search"
CHECKPOINT_DIR = SEARCH_ROOT / "checkpoints"
RESULTS_CSV = SEARCH_ROOT / "grid_results.csv"
SUMMARY_CSV = SEARCH_ROOT / "grid_summary.csv"
BEST_PARAMS_JSON = SEARCH_ROOT / "best_params.json"
FINAL_TXT_PATH = Path(DATA_ROOT) / f"{DATA_NAME}.txt"

SEARCH_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_FIELDS = [
    "config_id",
    "seed",
    "hidden_dim",
    "num_layers",
    "dropout",
    "lr",
    "weight_decay",
    "best_val_acc",
    "best_val_loss",
    "best_epoch",
    "epochs_ran",
    "checkpoint_path",
]
SUMMARY_FIELDS = [
    "config_id",
    "hidden_dim",
    "num_layers",
    "dropout",
    "lr",
    "weight_decay",
    "runs",
    "seeds",
    "val_accs",
    "mean_val_acc",
    "std_val_acc",
    "min_val_acc",
    "max_val_acc",
    "best_seed",
    "best_epoch",
    "best_checkpoint_path",
]


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if DETERMINISTIC:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        torch.backends.cudnn.benchmark = True


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def build_param_configs(param_grid):
    keys = list(param_grid.keys())
    configs = []
    for values in product(*(param_grid[key] for key in keys)):
        config = dict(zip(keys, values))
        config["bidirectional"] = BIDIRECTIONAL
        configs.append(config)
    return configs


def build_dataloaders(batch_size, seed):
    train_ds = TrainDataset(INDEX_PATH_TRAIN)
    val_ds = TrainDataset(INDEX_PATH_VAL)
    test_ds = TestDataset(INDEX_PATH_TEST)

    generator = torch.Generator()
    generator.manual_seed(seed)
    persistent_workers = NUM_WORKERS > 0

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=USE_CUDA,
        num_workers=NUM_WORKERS,
        persistent_workers=persistent_workers,
        worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
        generator=generator,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=USE_CUDA,
        num_workers=NUM_WORKERS,
        persistent_workers=persistent_workers,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=1,
        shuffle=False,
        pin_memory=USE_CUDA,
        num_workers=NUM_WORKERS,
        persistent_workers=persistent_workers,
    )
    return train_loader, val_loader, test_loader


def build_model(params):
    return EEGGRU(
        chans=CHANNELS,
        hidden_dim=int(params["hidden_dim"]),
        num_layers=int(params["num_layers"]),
        num_classes=CLASSES,
        dropout=float(params["dropout"]),
        bidirectional=bool(params.get("bidirectional", BIDIRECTIONAL)),
    ).to(device)


def checkpoint_path_for(config_id, seed):
    return CHECKPOINT_DIR / f"config_{config_id:03d}_seed_{seed}.pt"


def load_existing_results():
    if FORCE_RERUN or not RESULTS_CSV.exists():
        return []

    rows = []
    with RESULTS_CSV.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            checkpoint_path = Path(row.get("checkpoint_path", ""))
            if checkpoint_path.exists():
                rows.append(row)
    return rows


def normalize_result_row(row):
    normalized = dict(row)
    int_fields = ["config_id", "seed", "hidden_dim", "num_layers", "best_epoch", "epochs_ran"]
    float_fields = ["dropout", "lr", "weight_decay", "best_val_acc", "best_val_loss"]
    for field in int_fields:
        normalized[field] = int(normalized[field])
    for field in float_fields:
        normalized[field] = float(normalized[field])
    normalized["checkpoint_path"] = str(normalized["checkpoint_path"])
    return normalized


def write_results_csv(results):
    with RESULTS_CSV.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        writer.writeheader()
        writer.writerows(results)


def write_summary_csv(summary_rows):
    with SUMMARY_CSV.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=SUMMARY_FIELDS)
        writer.writeheader()
        writer.writerows(summary_rows)


def train_one_run(params, seed, config_id):
    set_global_seed(seed)
    train_loader, val_loader, _ = build_dataloaders(BATCH_SIZE, seed)

    model = build_model(params)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(params["lr"]),
        weight_decay=float(params["weight_decay"]),
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=8,
    )

    checkpoint_path = checkpoint_path_for(config_id, seed)
    best_val_acc = -1.0
    best_val_loss = math.inf
    best_epoch = 0
    bad_epochs = 0
    epochs_ran = 0

    print("\n" + "=" * 80)
    print(f"Config {config_id:03d} | seed={seed} | params={params}")

    for epoch in range(EPOCHS):
        epochs_ran = epoch + 1

        model.train()
        train_loss_sum = 0.0
        train_num = 0
        for data, label in train_loader:
            data = data.to(device, dtype=torch.float32, non_blocking=USE_CUDA)
            label = label.to(device, dtype=torch.long, non_blocking=USE_CUDA)

            optimizer.zero_grad(set_to_none=True)
            output = model(data)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            batch_size = label.size(0)
            train_loss_sum += loss.item() * batch_size
            train_num += batch_size

        epoch_train_loss = train_loss_sum / train_num

        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_num = 0
        with torch.no_grad():
            for val_data, val_label in val_loader:
                val_data = val_data.to(device, dtype=torch.float32, non_blocking=USE_CUDA)
                val_label = val_label.to(device, dtype=torch.long, non_blocking=USE_CUDA)

                val_output = model(val_data)
                val_loss = criterion(val_output, val_label)

                batch_size = val_label.size(0)
                val_loss_sum += val_loss.item() * batch_size
                val_num += batch_size
                val_pred = torch.argmax(val_output, dim=1)
                val_correct += (val_pred == val_label).sum().item()

        epoch_val_loss = val_loss_sum / val_num
        epoch_val_acc = val_correct / val_num
        scheduler.step(epoch_val_acc)

        improved = epoch_val_acc > best_val_acc + MIN_DELTA
        if improved:
            best_val_acc = epoch_val_acc
            best_val_loss = epoch_val_loss
            best_epoch = epoch + 1
            bad_epochs = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            bad_epochs += 1

        print(
            f"Epoch [{epoch + 1:03d}/{EPOCHS}] | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} | "
            f"Val Acc: {epoch_val_acc:.4f} | "
            f"Best Val Acc: {best_val_acc:.4f} @ {best_epoch:03d} | "
            f"Bad Epochs: {bad_epochs}/{PATIENCE}"
        )

        if bad_epochs >= PATIENCE:
            print(
                f"Early stopping at epoch {epoch + 1}; "
                f"best epoch was {best_epoch} with val acc {best_val_acc:.4f}."
            )
            break

    return {
        "config_id": config_id,
        "seed": seed,
        "hidden_dim": int(params["hidden_dim"]),
        "num_layers": int(params["num_layers"]),
        "dropout": float(params["dropout"]),
        "lr": float(params["lr"]),
        "weight_decay": float(params["weight_decay"]),
        "best_val_acc": float(best_val_acc),
        "best_val_loss": float(best_val_loss),
        "best_epoch": int(best_epoch),
        "epochs_ran": int(epochs_ran),
        "checkpoint_path": str(checkpoint_path),
    }


def aggregate_grid_results(results):
    grouped = {}
    for row in results:
        key = int(row["config_id"])
        grouped.setdefault(key, []).append(row)

    summary_rows = []
    for config_id, rows in grouped.items():
        rows = sorted(rows, key=lambda item: int(item["seed"]))
        val_accs = [float(row["best_val_acc"]) for row in rows]
        seeds = [int(row["seed"]) for row in rows]
        mean_val_acc = float(np.mean(val_accs))
        std_val_acc = float(np.std(val_accs, ddof=0))
        min_val_acc = float(np.min(val_accs))
        max_val_acc = float(np.max(val_accs))
        best_run = max(rows, key=lambda row: float(row["best_val_acc"]))

        summary_rows.append({
            "config_id": config_id,
            "hidden_dim": int(best_run["hidden_dim"]),
            "num_layers": int(best_run["num_layers"]),
            "dropout": float(best_run["dropout"]),
            "lr": float(best_run["lr"]),
            "weight_decay": float(best_run["weight_decay"]),
            "runs": len(rows),
            "seeds": json.dumps(seeds),
            "val_accs": json.dumps([round(acc, 8) for acc in val_accs]),
            "mean_val_acc": mean_val_acc,
            "std_val_acc": std_val_acc,
            "min_val_acc": min_val_acc,
            "max_val_acc": max_val_acc,
            "best_seed": int(best_run["seed"]),
            "best_epoch": int(best_run["best_epoch"]),
            "best_checkpoint_path": str(best_run["checkpoint_path"]),
        })

    summary_rows = sorted(
        summary_rows,
        key=lambda row: (
            -float(row["mean_val_acc"]),
            -float(row["min_val_acc"]),
            float(row["std_val_acc"]),
            -float(row["max_val_acc"]),
            int(row["config_id"]),
        ),
    )
    best_summary = summary_rows[0]
    best_run = max(
        [row for row in results if int(row["config_id"]) == int(best_summary["config_id"])],
        key=lambda row: float(row["best_val_acc"]),
    )
    return summary_rows, best_summary, best_run


def load_state_dict_safely(checkpoint_path):
    try:
        return torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(checkpoint_path, map_location=device)


def write_test_txt(best_run):
    params = {
        "hidden_dim": int(best_run["hidden_dim"]),
        "num_layers": int(best_run["num_layers"]),
        "dropout": float(best_run["dropout"]),
        "bidirectional": BIDIRECTIONAL,
    }
    set_global_seed(int(best_run["seed"]))
    _, _, test_loader = build_dataloaders(BATCH_SIZE, int(best_run["seed"]))

    model = build_model(params)
    checkpoint_path = Path(best_run["checkpoint_path"])
    model.load_state_dict(load_state_dict_safely(checkpoint_path))
    model.eval()

    all_test_labels = []
    with torch.no_grad():
        for test_data in test_loader:
            test_data = test_data.to(device, dtype=torch.float32, non_blocking=USE_CUDA)
            test_output = model(test_data)
            test_pred = torch.argmax(test_output, dim=1)
            all_test_labels.extend(test_pred.cpu().tolist())

    with FINAL_TXT_PATH.open("w", encoding="utf-8") as f:
        for label in all_test_labels:
            f.write(f"{int(label)}\n")

    backup_txt_path = SEARCH_ROOT / (
        f"{DATA_NAME}_best_config_{int(best_run['config_id']):03d}_"
        f"seed_{int(best_run['seed'])}.txt"
    )
    with backup_txt_path.open("w", encoding="utf-8") as f:
        for label in all_test_labels:
            f.write(f"{int(label)}\n")

    return {
        "label_count": len(all_test_labels),
        "final_txt_path": str(FINAL_TXT_PATH),
        "backup_txt_path": str(backup_txt_path),
    }


def save_best_params(best_summary, best_run, txt_info):
    payload = {
        "data_name": DATA_NAME,
        "selection_rule": [
            "max mean_val_acc",
            "max min_val_acc",
            "min std_val_acc",
            "max single-run best_val_acc",
        ],
        "best_params": {
            "hidden_dim": int(best_summary["hidden_dim"]),
            "num_layers": int(best_summary["num_layers"]),
            "dropout": float(best_summary["dropout"]),
            "lr": float(best_summary["lr"]),
            "weight_decay": float(best_summary["weight_decay"]),
            "bidirectional": BIDIRECTIONAL,
            "batch_size": BATCH_SIZE,
            "label_smoothing": LABEL_SMOOTHING,
        },
        "summary": {
            "config_id": int(best_summary["config_id"]),
            "runs": int(best_summary["runs"]),
            "seeds": json.loads(best_summary["seeds"]),
            "val_accs": json.loads(best_summary["val_accs"]),
            "mean_val_acc": float(best_summary["mean_val_acc"]),
            "std_val_acc": float(best_summary["std_val_acc"]),
            "min_val_acc": float(best_summary["min_val_acc"]),
            "max_val_acc": float(best_summary["max_val_acc"]),
        },
        "best_run_for_txt": {
            "seed": int(best_run["seed"]),
            "best_epoch": int(best_run["best_epoch"]),
            "best_val_acc": float(best_run["best_val_acc"]),
            "checkpoint_path": str(best_run["checkpoint_path"]),
        },
        "outputs": txt_info,
    }
    with BEST_PARAMS_JSON.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    return payload


param_configs = build_param_configs(PARAM_GRID)
expected_runs = len(param_configs) * len(SEEDS)
print(f"Grid configs: {len(param_configs)} | seeds/config: {len(SEEDS)} | total runs: {expected_runs}")
print(f"Results directory: {SEARCH_ROOT}")

existing_results = [normalize_result_row(row) for row in load_existing_results()]
completed = {(row["config_id"], row["seed"]) for row in existing_results}
results = existing_results.copy()

if RUN_GRID_SEARCH:
    for config_index, params in enumerate(param_configs, start=1):
        for seed in SEEDS:
            checkpoint_path = checkpoint_path_for(config_index, seed)
            if not FORCE_RERUN and (config_index, seed) in completed and checkpoint_path.exists():
                print(f"Skip existing run: config={config_index:03d}, seed={seed}")
                continue

            result = train_one_run(params, seed, config_index)
            results = [
                row for row in results
                if not (row["config_id"] == config_index and row["seed"] == seed)
            ]
            results.append(result)
            completed.add((config_index, seed))
            write_results_csv(sorted(results, key=lambda row: (row["config_id"], row["seed"])))
else:
    print("RUN_GRID_SEARCH is False; using existing grid_results.csv only.")

if not results:
    raise RuntimeError("No grid search results are available. Set RUN_GRID_SEARCH=True or provide grid_results.csv.")

results = sorted(results, key=lambda row: (row["config_id"], row["seed"]))
write_results_csv(results)
summary_rows, best_summary, best_run = aggregate_grid_results(results)
write_summary_csv(summary_rows)
txt_info = write_test_txt(best_run)
best_params_payload = save_best_params(best_summary, best_run, txt_info)

# Keep the best model and test loader available for optional follow-up cells.
model = build_model(best_params_payload["best_params"])
model.load_state_dict(load_state_dict_safely(best_run["checkpoint_path"]))
model.eval()
_, _, test_loader = build_dataloaders(BATCH_SIZE, int(best_run["seed"]))

print("\n" + "-" * 80)
print("Best parameter group")
print(json.dumps(best_params_payload["best_params"], ensure_ascii=False, indent=2))
print(f"Validation accuracies: {best_params_payload['summary']['val_accs']}")
print(
    "mean_val_acc={mean:.4f} | std_val_acc={std:.4f} | "
    "best_seed={seed} | best_epoch={epoch}".format(
        mean=best_params_payload["summary"]["mean_val_acc"],
        std=best_params_payload["summary"]["std_val_acc"],
        seed=best_params_payload["best_run_for_txt"]["seed"],
        epoch=best_params_payload["best_run_for_txt"]["best_epoch"],
    )
)
print(f"Best params saved to: {BEST_PARAMS_JSON}")
print(f"Grid results saved to: {RESULTS_CSV}")
print(f"Grid summary saved to: {SUMMARY_CSV}")
print(f"Saved {txt_info['label_count']} labels to: {txt_info['final_txt_path']}")
print(f"Backup txt saved to: {txt_info['backup_txt_path']}")


Grid configs: 48 | seeds/config: 5 | total runs: 240
Results directory: /root/autodl-tmp/MLproject/course_project/SEED/grid_search

Config 001 | seed=42 | params={'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.001, 'bidirectional': True}


/root/miniconda3/lib/python3.12/site-packages/torch/autograd/graph.py:829: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch [001/100] | Train Loss: 1.1137 | Val Loss: 1.0981 | Val Acc: 0.3378 | Best Val Acc: 0.3378 @ 001 | Bad Epochs: 0/20
Epoch [002/100] | Train Loss: 1.1078 | Val Loss: 1.1309 | Val Acc: 0.3333 | Best Val Acc: 0.3378 @ 001 | Bad Epochs: 1/20
Epoch [003/100] | Train Loss: 1.1046 | Val Loss: 1.0977 | Val Acc: 0.3556 | Best Val Acc: 0.3556 @ 003 | Bad Epochs: 0/20
Epoch [004/100] | Train Loss: 1.0956 | Val Loss: 1.0978 | Val Acc: 0.3511 | Best Val Acc: 0.3556 @ 003 | Bad Epochs: 1/20
Epoch [005/100] | Train Loss: 1.1014 | Val Loss: 1.1019 | Val Acc: 0.3800 | Best Val Acc: 0.3800 @ 005 | Bad Epochs: 0/20
Epoch [006/100] | Train Loss: 1.0982 | Val Loss: 1.0973 | Val Acc: 0.3800 | Best Val Acc: 0.3800 @ 005 | Bad Epochs: 1/20
Epoch [007/100] | Train Loss: 1.0851 | Val Loss: 1.1000 | Val Acc: 0.3333 | Best Val Acc: 0.3800 @ 005 | Bad Epochs: 2/20
Epoch [008/100] | Train Loss: 1.0784 | Val Loss: 1.1086 | Val Acc: 0.3089 | Best Val Acc: 0.3800 @ 005 | Bad Epochs: 3/20
Epoch [009/100] | Train 

## 7) 查看最佳参数与最终 txt 输出
这一格读取网格搜索保存的最佳参数文件，打印最佳参数、验证集统计、用于生成 txt 的 seed/checkpoint，以及最终 txt 文件路径。


In [ ]:
# -------------------------
# Show the selected parameters and output files
# -------------------------
from pathlib import Path
import json

best_params_path = Path(DATA_ROOT) / "grid_search" / "best_params.json"
if not best_params_path.exists():
    raise FileNotFoundError(
        f"Missing {best_params_path}. Run the grid-search cell first."
    )

with best_params_path.open("r", encoding="utf-8") as f:
    best_params_payload = json.load(f)

print("Best parameters:")
print(json.dumps(best_params_payload["best_params"], ensure_ascii=False, indent=2))
print("\nValidation summary:")
print(json.dumps(best_params_payload["summary"], ensure_ascii=False, indent=2))
print("\nBest run used for txt:")
print(json.dumps(best_params_payload["best_run_for_txt"], ensure_ascii=False, indent=2))
print("\nOutputs:")
print(json.dumps(best_params_payload["outputs"], ensure_ascii=False, indent=2))
